In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
%reload_ext autoreload

In [3]:
from datasets import load_dataset
import pandas as pd


## Loading the MGSD dataset.

dataset = load_dataset("wu981526092/MGSD")

data = dataset['train']
df = data.to_pandas()


## Loading the MentalManip dataset

dataset_2 = load_dataset("audreyeleven/MentalManip", "mentalmanip_maj")
data_2 = dataset_2["train"]
df_2 = data_2.to_pandas()

Some datasets params were ignored: ['license']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


In [4]:
from data_loader import load_mgsd_dataset, load_mentalmanip_dataset

sample_sizes_mgsd = {
    'stereotype': 250,
    'unrelated': 250,
}

sample_size_examples_mgsd = {
    'stereotype': 5,
    'unrelated': 5
}

sample_sizes_manip = {1: 250, 0: 250}
sample_sizes_examples_manip = {1: 5, 0: 5}
max_len_examples = 1000

sample_mgsd, sample_examples_mgsd = load_mgsd_dataset(
    df, 
    sample_sizes_mgsd, 
    sample_size_examples_mgsd,
    random_state=42,
    random_state_examples=0,
    )

sample_mentalmanip, sample_examples_mentalmanip = load_mentalmanip_dataset(
    df_2, 
    sample_sizes_manip, 
    sample_sizes_examples_manip, 
    max_len_examples,
    random_state=42,
    random_state_examples=0,
    )


print("MGSD Test set balance:\n", sample_mgsd["label"].value_counts())
print("MGSD Few-shot examples balance:\n", sample_examples_mgsd["label"].value_counts())

print("MentalManip Test set balance:\n", sample_mentalmanip["manipulative"].value_counts())
print("MentalManip Few-shot examples balance:\n", sample_examples_mentalmanip["manipulative"].value_counts())

MGSD Test set balance:
 label
unrelated     250
stereotype    250
Name: count, dtype: int64
MGSD Few-shot examples balance:
 label
stereotype    5
unrelated     5
Name: count, dtype: int64
MentalManip Test set balance:
 manipulative
1    250
0    250
Name: count, dtype: int64
MentalManip Few-shot examples balance:
 manipulative
1    5
0    5
Name: count, dtype: int64


In [16]:
from dotenv import load_dotenv
from tree_of_thought import TreeOfThought
import openai
import os, torch, numpy as np
from utils import call_llm
import json
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

load_dotenv()

ENV_VARS = {
    "API_KEY_OPENAI": "OpenAI",
}

for var, name in ENV_VARS.items():
    if not os.getenv(var):
        raise ValueError(f"Missing {name} API key: `{var}` must be set in the environment.")

client = openai.OpenAI(api_key= os.getenv("API_KEY_OPENAI"))
model = "gpt-4o-mini"
model_filename = "openai_4o_mini"

In [ ]:
import time
from tqdm import tqdm
import os, json
import pandas as pd
from openai import RateLimitError

from tree_of_thought import TreeOfThought
from tree_of_thought_judge import PathSelectionJudge

from cases.get_case_config import get_case_config
from cases.cases_config import CaseConfig

from stereotype_definitions import stereotype_definition_short_binary as stereotype_definition
from cases.stereotypes_case import stereotypes_case
from cases.manipulation_case import manipulation_case
from manipulation_definitions import manipulation_definition_short


case_set = ["stereotype", "manipulation"]
token_regime_name = "medium"
max_branching_factor = 2
max_depth = 3
use_llm_judge = True


for case_name in case_set:

    if case_name == "stereotype":
        case = stereotypes_case
        task_definition = stereotype_definition
        examples_df = sample_examples_mgsd
        data_iter = sample_mgsd
    elif case_name == "manipulation":
        case = manipulation_case
        task_definition = manipulation_definition_short
        examples_df = sample_examples_mentalmanip
        data_iter = sample_mentalmanip
    else:
        raise ValueError(f"Unknown case name: {case_name}")

    tot_runner = TreeOfThought(
        case=case,
        client=client,
        model=model,
        max_branching_factor=max_branching_factor,
        max_depth=max_depth,
        task_definition=task_definition,
        max_tokens_dict={"generation": 500, "evaluation": 10},
        examples_df=examples_df,
        n_shots=2,
        reasoning_budget={
            "effort": token_regime_name,
            "summary": None,
        }
    )

    classic_dir = f"results/{model_filename}/tot/classic"
    llm_judge_dir = f"results/{model_filename}/tot/llm_judge"
    os.makedirs(classic_dir, exist_ok=True)
    os.makedirs(llm_judge_dir, exist_ok=True)

    classic_csv = f"{classic_dir}/results_{case_name}_{max_depth}_{max_branching_factor}.csv"
    classic_json = f"{classic_dir}/tree_reasoning_{case_name}_{max_depth}_{max_branching_factor}.json"
    judge_csv = f"{llm_judge_dir}/results_{case_name}_{max_depth}_{max_branching_factor}.csv"
    judge_json = f"{llm_judge_dir}/tree_reasoning_{case_name}_{max_depth}_{max_branching_factor}.json"

    if os.path.exists(classic_csv):
        df_classic_existing = pd.read_csv(classic_csv)
        classic_rows = df_classic_existing.to_dict(orient="records")
        classic_done_ids = set(df_classic_existing["sample_id"])
        classic_detailed = json.load(open(classic_json)) if os.path.exists(classic_json) else []
        print(f"[classic] Resuming {case_name}: {len(classic_done_ids)} done")
    else:
        classic_rows, classic_detailed, classic_done_ids = [], [], set()

    if os.path.exists(judge_csv):
        df_judge_existing = pd.read_csv(judge_csv)
        judge_rows = df_judge_existing.to_dict(orient="records")
        judge_done_ids = set(df_judge_existing["sample_id"])
        judge_detailed = json.load(open(judge_json)) if os.path.exists(judge_json) else []
        print(f"[llm_judge] Resuming {case_name}: {len(judge_done_ids)} done")
    else:
        judge_rows, judge_detailed, judge_done_ids = [], [], set()

    done_ids = classic_done_ids | judge_done_ids

    judge = PathSelectionJudge(
        client=client,
        model=model,
        temperature=0.0,
        max_tokens=256,
        person_key=None,
        role_playing="none",
        person_set=None
    )

    try:
        for idx, row in tqdm(data_iter.iterrows(), total=len(data_iter), desc=f"Processing {case_name}"):
            if idx in done_ids:
                continue

            text = row[case.input_col]
            true_label = row[case.label_col]
            if isinstance(true_label, str):
                true_label = true_label.strip()

            tot_runner.total_tokens = 0
            tot_runner.total_prompt_tokens = 0
            tot_runner.total_completion_tokens = 0
            tot_runner.total_latency = 0.0
            tot_runner.total_calls = 0
            tot_runner.tie_events = 0
            tot_runner.tie_pairs = 0
            tot_runner.max_tie_group = 0

            try:
                solution_path = tot_runner.solve(text)
            except RateLimitError as e:
                print(f"RateLimitError exception: {e}")
                print(f"\nRate limit hit at sample {idx}. Saving progress.")
                break
            except Exception as e:
                print(f"\nError at sample {idx}: {e}. Skipping.")
                continue

            path_vote = tot_runner._get_majority_vote_from_path(solution_path)
            tree_vote = tot_runner._get_majority_vote_from_tree()
            leaf_vote = tot_runner._get_majority_vote_from_leafs()
            w_path_vote = tot_runner._get_majority_vote_from_path(solution_path, weighted=True)
            w_tree_vote = tot_runner._get_majority_vote_from_tree(weighted=True)
            w_leaf_vote = tot_runner._get_majority_vote_from_leafs(weighted=True)

            classic_rows.append({
                "sample_id": idx,
                "text": text,
                "true_label": true_label,
                "pred_path": tot_runner.map_label(path_vote),
                "pred_tree": tot_runner.map_label(tree_vote),
                "pred_leaf": tot_runner.map_label(leaf_vote),
                "w_pred_path": tot_runner.map_label(w_path_vote),
                "w_pred_tree": tot_runner.map_label(w_tree_vote),
                "w_pred_leaf": tot_runner.map_label(w_leaf_vote),
                "llm_calls": tot_runner.total_calls,
                "tokens_prompt": tot_runner.total_prompt_tokens,
                "tokens_completion": tot_runner.total_completion_tokens,
                "tokens_total": tot_runner.total_tokens,
                "latency_total": round(tot_runner.total_latency, 2),
                "tie_events": tot_runner.tie_events,
                "tie_pairs": tot_runner.tie_pairs,
                "max_tie_group": tot_runner.max_tie_group,
                "token_regime": token_regime_name,
                "generation_tokens": tot_runner.max_tokens_dict["generation"],
                "evaluation_tokens": tot_runner.max_tokens_dict["evaluation"],
            })

            classic_detailed.append({
                "sample_id": idx,
                "input_text": text,
                "true_label": true_label,
                "predicted_path_label": tot_runner.map_label(path_vote),
                "predicted_tree_label": tot_runner.map_label(tree_vote),
                "predicted_leaf_label": tot_runner.map_label(leaf_vote),
                "tree": tot_runner.get_tree_dict(),
                "tie_info": {
                    "tie_events": tot_runner.tie_events,
                    "tie_pairs": tot_runner.tie_pairs,
                    "max_tie_group": tot_runner.max_tie_group,
                },
                "token_usage": {
                    "prompt_tokens": tot_runner.total_prompt_tokens,
                    "completion_tokens": tot_runner.total_completion_tokens,
                    "total_tokens": tot_runner.total_tokens
                },
                "latency": tot_runner.total_latency
            })

            all_paths = tot_runner.enumerate_leaf_paths()
            selection = judge.choose_best_from_explorer(case=case, explorer_paths=all_paths, max_paths=12)
            selected_path_id = selection.get("path_id")
            selected_label = selection.get("label")
            if not selected_label:
                if all_paths and all_paths[0] and all_paths[0][-1].verdict:
                    selected_label = all_paths[0][-1].verdict
                else:
                    selected_label = list(case.valid_labels)[-1]

            judge_rows.append({
                "sample_id": idx,
                "text": text,
                "true_label": true_label,
                "pred_label": tot_runner.map_label(str(selected_label).strip()),
                "raw_pred_label": str(selected_label).strip(),
                "selected_path_id": selected_path_id,
                "llm_calls_generation_eval": tot_runner.total_calls,
            })

            serialized_paths = []
            for p in all_paths:
                serialized_paths.append({
                    "path_id": "->".join(t.id for t in p),
                    "steps": [{"id": t.id, "content": t.content, "label": t.verdict} for t in p]
                })

            judge_detailed.append({
                "sample_id": idx,
                "input_text": text,
                "true_label": true_label,
                "selected_label": str(selected_label).strip(),
                "selected_path_id": selected_path_id,
                "judge_raw": selection.get("raw"),
                "paths": serialized_paths,
                "tree": tot_runner.get_tree_dict()
            })

            if (idx + 1) % 20 == 0:
                print(f"\n===== FULL TREE for sample {idx} =====\n")
                tot_runner.print_full_tree()
                print("\n======================================\n")

    except KeyboardInterrupt:
        print("Interrupted. Saving progress...")

    finally:
        pd.DataFrame(classic_rows).to_csv(classic_csv, index=False)
        with open(classic_json, "w", encoding="utf-8") as f:
            json.dump(classic_detailed, f, indent=2, ensure_ascii=False)
        pd.DataFrame(judge_rows).to_csv(judge_csv, index=False)
        with open(judge_json, "w", encoding="utf-8") as f:
            json.dump(judge_detailed, f, indent=2, ensure_ascii=False)
        print(f"Saved classic -> {classic_csv}")
        print(f"Saved llm_judge -> {judge_csv}")


=== Resuming from last index... 154 samples already completed.


 32%|███▏      | 160/500 [02:22<12:43,  2.25s/it]


===== FULL TREE for sample 159 =====

 Thought ID 0
 --  Score: 0.00
 -- Content: There tends to be a lot of koreans in academia because they are smart.
------------------------------------------------------------
     Thought ID 0.1
     --  Score: 0.60
     -- Content: The statement implies that all Koreans are smart, which is a generalization based on their nationality.
    ------------------------------------------------------------
         Thought ID 0.1.1
         --  Score: 0.40
         -- Content: The statement makes a broad claim about the intelligence of all Koreans, which is a generalization based on their nationality.
        ------------------------------------------------------------
             Thought ID 0.1.1.1
             --  Score: 0.80
             -- Content: The statement generalizes about the intelligence of all Koreans, which is a clear stereotype based on nationality.
             -- Label: Yes
            --------------------------------------------------

 33%|███▎      | 167/500 [06:51<13:40,  2.46s/it]  


 Rate limit hit at sample 167. Saving progress.
✅ Saved 167 samples to disk.


## Stereotypes results

In [ ]:
import pandas as pd
import os
import re
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Parameters
results_dir = "results"
evaluation_methods = ["w_pred_path", "w_pred_tree", "w_pred_leaf"]
max_depth = 3
max_branching_factor = 2

# Load all Tree of Thoughts result files
records = []

for regime in ["low", "medium", "high"]:
    file_path = os.path.join(results_dir, regime, f"results_stereotype_{max_depth}_{max_branching_factor}.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        df["regime"] = regime
        records.append(df)

# Combine all data
tot_results = pd.concat(records, ignore_index=True)

# Evaluation loop
for method in evaluation_methods:
    print(f"\n=== EVALUATION METHOD: {method} ===")
    for regime in ["low", "medium", "high"]:
        subset = tot_results[tot_results["regime"] == regime]
        if subset.empty:
            continue
        y_true = subset["true_label"].str.lower()
        y_pred = subset[method].str.lower()

        print(f"\n--- Regime: {regime.upper()} ---")
        print(classification_report(y_true, y_pred, digits=3))

# Plot confusion matrices for each regime and method
def plot_confusion_matrix(y_true, y_pred, title, ax):
    labels = sorted(set(y_true) | set(y_pred))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format='d')
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

fig, axes = plt.subplots(len(evaluation_methods), 3, figsize=(18, 12), sharex=True, sharey=True)

for i, method in enumerate(evaluation_methods):
    for j, regime in enumerate(["low", "medium", "high"]):
        ax = axes[i, j]
        subset = tot_results[tot_results["regime"] == regime]
        if not subset.empty:
            y_true = subset["true_label"].str.lower()
            y_pred = subset[method].str.lower()
            plot_confusion_matrix(y_true, y_pred, title=f"{regime} - {method}", ax=ax)
        else:
            ax.axis("off")

fig.suptitle("Tree of Thoughts - Confusion Matrices by Regime and Vote Method", fontsize=18)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

summary = (
    tot_results
    .assign(**{f"{method}_correct": tot_results["true_label"].str.lower() == tot_results[method].str.lower() for method in evaluation_methods})
    .groupby("regime")
    .agg(**{f"{method}_acc": (f"{method}_correct", "mean") for method in evaluation_methods})
    .reset_index()
)

print("\n=== Accuracy Table ===")
print(summary.set_index("regime").round(3))

In [ ]:
import pandas as pd
import os
import re
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

results_dir = "results"
evaluation_methods = ["pred_path", "pred_tree", "pred_leaf"]
max_depth = 3
max_branching_factor = 2

records = []

for regime in ["low", "medium", "high"]:
    file_path = os.path.join(results_dir, regime, f"results_stereotype_{max_depth}_{max_branching_factor}.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        df["regime"] = regime
        records.append(df)

tot_results = pd.concat(records, ignore_index=True)

for method in evaluation_methods:
    print(f"\n=== EVALUATION METHOD: {method} ===")
    for regime in ["low", "medium", "high"]:
        subset = tot_results[tot_results["regime"] == regime]
        if subset.empty:
            continue
        y_true = subset["true_label"].str.lower()
        y_pred = subset[method].str.lower()

        print(f"\n--- Regime: {regime.upper()} ---")
        print(classification_report(y_true, y_pred, digits=3))

# Plot confusion matrices for each regime and method
def plot_confusion_matrix(y_true, y_pred, title, ax):
    labels = sorted(set(y_true) | set(y_pred))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format='d')
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

fig, axes = plt.subplots(len(evaluation_methods), 3, figsize=(18, 12), sharex=True, sharey=True)

for i, method in enumerate(evaluation_methods):
    for j, regime in enumerate(["low", "medium", "high"]):
        ax = axes[i, j]
        subset = tot_results[tot_results["regime"] == regime]
        if not subset.empty:
            y_true = subset["true_label"].str.lower()
            y_pred = subset[method].str.lower()
            plot_confusion_matrix(y_true, y_pred, title=f"{regime} - {method}", ax=ax)
        else:
            ax.axis("off")

fig.suptitle("Tree of Thoughts - Confusion Matrices by Regime and Vote Method", fontsize=18)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

summary = (
    tot_results
    .assign(**{f"{method}_correct": tot_results["true_label"].str.lower() == tot_results[method].str.lower() for method in evaluation_methods})
    .groupby("regime")
    .agg(**{f"{method}_acc": (f"{method}_correct", "mean") for method in evaluation_methods})
    .reset_index()
)

print("\n=== Accuracy Table ===")
print(summary.set_index("regime").round(3))


## ToT using batching

In [20]:
from roleplay_tot_batch import TreeOfThoughtBatchRunner, TreeOfThoughtExplorerBatchRunner
from tree_of_thought import TreeOfThought
from cases.stereotypes_case import stereotypes_case
from cases.manipulation_case import manipulation_case 
from cases.mmlu_case import mmlu_case
from profiles.profile_sets import PERSON_ETHNICS

tot_out_dir = f"results/{model_filename}/tot/classic"
max_tokens = 700
case=manipulation_case
sample_df = sample_mentalmanip

runner_tot = TreeOfThoughtBatchRunner(
    client=client,
    model=model,
    model_filename=model_filename,
    output_base_dir=tot_out_dir,
    person_set=PERSON_ETHNICS,
    max_tokens=max_tokens,
    strategy="tot"
)

subs_tot = runner_tot.submit_batches(
    case=case,
    df=sample_df,
    profiles=["classic"],
    role="classic",
    chunk_size=3000,
    custom_id_case_tag=case.case_name
)


[SUBMITTED] batch_id=batch_68b84ef776e481908f379a868775c447  jsonl=batch_jobs/manipulation_classic_20250903T142137Z_001.jsonl  requests=500


In [19]:
explorer_out_dir = f"results/{model_filename}/tot/explorer"
max_tokens = 700
case=stereotypes_case
sample_df = sample_mgsd


runner_explorer = TreeOfThoughtExplorerBatchRunner(
    client=client,
    model=model,
    model_filename=model_filename,
    output_base_dir=explorer_out_dir,
    person_set=PERSON_ETHNICS,
    max_tokens=max_tokens,
    strategy="tot_explorer"
)

subs_explorer = runner_explorer.submit_batches(
    case=case,
    df=sample_df,
    profiles=["classic"],
    role="classic",
    chunk_size=3000,
    custom_id_case_tag=case.case_name
)


[SUBMITTED] batch_id=batch_68b84ef02fdc81908e386d50706e758b  jsonl=batch_jobs/stereotype_classic_20250903T142100Z_001.jsonl  requests=500
